In [1]:
import torch
from torch import nn
import os
import pandas as pd
from torch.utils.data import  DataLoader
from torchvision import transforms,datasets,models
import torch.optim as optim
from PIL import Image
import numpy as np

In [ ]:
DATA_DIR =  r"F:\Work\ORANGE MODELS\Bone\Train" # Make sure that your folder has two folders, each containing the respective class images, as they will be used for labeling #
MODEL_PATH = "Bone_Fracture_detector.pth" # Here the learned model will be saved, you can change the name if you want #
BATCH_SIZE = 32
IMG_SIZE = 224
EPOCHS = 5 # You can increase this for better performance, but it will take more time to train #

In [17]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                                          [0.229, 0.224, 0.225])
])

In [18]:
train_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [19]:
class_names = train_dataset.classes  #just to check the class names#
print("Classes:", class_names)

Classes: ['Normal', 'You Have a Fracutre']


In [20]:
# Use pre-trained ResNet18 for better performance
model = models.resnet18(pretrained=True)
# Freeze all layers except the last few
for param in model.parameters():
    param.requires_grad = False
# Unfreeze the layer4 and fc
for param in model.layer4.parameters():
    param.requires_grad = True
# Replace the final layer
model.fc = nn.Linear(model.fc.in_features, 2) # 2 classes: fracture and Normal

In [21]:
# Use this if you have a powerful GPU, otherwise it will run on CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
model = model.to(device)
device

device(type='cpu')

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# To check for bad images in the dataset

data_dir_for_checking = r'' # Put the path to your dataset that you want to check for bad images #

bad_files = []

for root, dirs, files in os.walk(data_dir_for_checking):
    for file in files:
        path = os.path.join(root, file)

        try:
            img = Image.open(path)
            img.load()   # Fully load image

        except Exception as e:
            print(f"Bad image: {path}")
            print(e)
            bad_files.append(path)

print("\nDone")


Done


In [23]:
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct = 0.0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += torch.sum(preds == labels).item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset) * 100
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

Epoch 1/5 - Loss: 0.1087, Acc: 95.81%
Epoch 2/5 - Loss: 0.0321, Acc: 98.87%
Epoch 3/5 - Loss: 0.0190, Acc: 99.38%
Epoch 4/5 - Loss: 0.0177, Acc: 99.41%
Epoch 5/5 - Loss: 0.0137, Acc: 99.51%


In [24]:
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
image_path = r"F:\Work\ORANGE MODELS\Bone\New folder\Bone_Fracture_Binary_Classification\Bone_Fracture_Binary_Classification\test\fractured\4-rotated3-rotated3-rotated1.jpg"# Put the image path of your image that you want to evaluate to test whether it works correctly #
image = Image.open(image_path).convert('RGB')
image = transform(image).unsqueeze(0).to(device)
with torch.no_grad():
    outputs = model(image)
    print("Outputs:", outputs)
    _, predicted = torch.max(outputs, 1)
    print("Predicted index:", predicted.item())
    predicted_class = class_names[predicted.item()]

print(f"Predicted: {predicted_class}")
print("Class names:", class_names)

Outputs: tensor([[-12.1245,  10.3880]])
Predicted index: 1
Predicted: You Have a Fracutre
Class names: ['Normal', 'You Have a Fracutre']


In [25]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, MODEL_PATH)

print(" Model saved at", MODEL_PATH)

 Model saved at Bone_Fracture_detector.pth


In [4]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# Load the pre-trained ResNet18 model [Load the model architecture first, then load the weights]
model = models.resnet18(pretrained=True)
# Freeze all layers except the last few
for param in model.parameters():
    param.requires_grad = False
# Unfreeze the layer4 and fc
for param in model.layer4.parameters():
    param.requires_grad = True
# Replace the final layer
model.fc = nn.Linear(model.fc.in_features, 2)

# Load the saved model
MODEL_PATH = "Bone_Fracture_detector.pth"
checkpoint = torch.load(MODEL_PATH)
model.load_state_dict(checkpoint["model_state"])
class_names = checkpoint["class_names"]
model.eval()

# Specify the image path here
image_path = r"F:\Work\ORANGE MODELS\Bone\New folder (6)\Bone_Fracture_Dataset\val\not fractured\3-rotated2-rotated3-rotated1-rotated1.jpg"  # Change this to the image path you want to use

# Prepare and predict on the image
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                                          [0.229, 0.224, 0.225])
])
image = Image.open(image_path).convert('RGB')
image = transform(image).unsqueeze(0)
with torch.no_grad():
    outputs = model(image)
    print("Outputs:", outputs)
    _, predicted = torch.max(outputs, 1)
    print("Predicted index:", predicted.item())
    predicted_class = class_names[predicted.item()]

print(f"Predicted: {predicted_class}")

Outputs: tensor([[ 5.6485, -5.6237]])
Predicted index: 0
Predicted: Normal


f:\Work\Python Learning\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
f:\Work\Python Learning\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
